# Gradient Extraction & Storing

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

#########################################
# 1. Define a simple CNN with 3 conv layers
#########################################
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # conv1: from 4x4 input → output remains 4x4
        self.conv1 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False)
        # conv2: from 4x4 → 2x2 (stride=2)
        self.conv2 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1, bias=False)
        # conv3: from 2x2 → 1x1 (stride=2)
        self.conv3 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1, bias=False)
        self.fcn = nn.Linear(1, 10, bias=False)  # not used for interaction

    def forward(self, x):
        
        a1 = (self.conv1(x))    # shape: (1,1,4,4)
        a2 = (self.conv2(a1))     # shape: (1,1,2,2)
        a3 = (self.conv3(a2))     # shape: (1,1,1,1)
        return a3

model = SimpleCNN()
criterion = nn.CrossEntropyLoss(reduction="none")
optimizer = optim.SGD(model.parameters(), lr=0.01)


#########################################
# 2. Register hooks to capture gradients
#########################################
activation_gradients = {}
gradient_flows = {}
def get_activation_grad(name, connet2name=None):
    def hook(module, grad_input, grad_output):
        if name is not None:
            # Lấy grad_out: shape [N, C_out, H_out, W_out]
            grad_out = grad_output[0].detach()
            N, C_out, H_out, W_out = grad_out.shape
            
            # Lấy weight của module: shape [C_out, C_in, kH, kW]
            weight = module.weight  
            C_out_w, C_in, kH, kW = weight.shape
            assert C_out == C_out_w, "Mismatch in output channels."

            # Chuyển weight thành dạng ma trận: [C_out, C_in*kH*kW]
            weight_reshaped = weight.view(C_out, -1)
            
            # Reshape grad_out thành [N, C_out, Len_out] với Len_out = H_out * W_out
            Len_out = H_out * W_out
            grad_out_reshaped = grad_out.view(N, C_out, Len_out)

            # Tính grad_input_cols: [N, C_in*kH*kW, Len_out]
            grad_input_cols = torch.matmul(weight_reshaped.t(), grad_out_reshaped)
            
            # Giả sử batch size N=1
            grad_input_cols = grad_input_cols[0]  # [C_in*kH*kW, Len_out]

            # Lấy kích thước input từ grad_input[0]: [N, C_in, H_in, W_in]
            H_in, W_in = grad_input[0].shape[2:]
            Len_in = H_in * W_in

            # Xây dựng ánh xạ từ các patch đến các vị trí trên input:
            # Tạo tensor chứa các chỉ số của các ô input, shape: [1, 1, H_in, W_in]
            input_indices = torch.arange(Len_in, device=grad_out.device).view(1, 1, H_in, W_in).float()+1
            # Sử dụng F.unfold để lấy ma trận ánh xạ, shape: [C_in*kH*kW, Len_out]
            idx_map = (F.unfold(input_indices, kernel_size=module.kernel_size, 
                                dilation=module.dilation, padding=module.padding, stride=module.stride)[0])

            # Khởi tạo gradient_flows với kích thước (Len_in, Len_out)
            gradient_flow = torch.zeros(Len_in+1, Len_out, device=grad_out.device)
            # Sử dụng scatter_add_ để cộng các giá trị từ grad_input_cols vào gradient_flows
            # Cho mỗi phần tử tại vị trí (p, j) trong grad_input_cols, ta cộng vào gradient_flows tại (idx_map[p,j], j)
            gradient_flow.scatter_add_(0, idx_map.long(), grad_input_cols)
            gradient_flow = (gradient_flow[1:,:])
            # print((grad_input[0]).cpu().numpy().reshape(-1))
            # print(gradient_flow.cpu().numpy().sum(axis=-1,keepdims=False))
            assert np.abs(gradient_flow.cpu().numpy().sum(axis=-1,keepdims=False)-(grad_input[0]).cpu().numpy().reshape(-1)).sum() < 1e-7, "Mismatch in gradient values."
            gradient_flow = torch.abs(gradient_flow)
            gradient_flow[gradient_flow>1e-5] = 1.
            gradient_flow[gradient_flow<=1e-5] = .99

            # Lưu kết quả vào activation_gradients
            gradient_flows[(name, connet2name)] = gradient_flow.cpu().numpy()  # kích thước: (Len_in, Len_out)
            
            activation_gradients[name] = (gradient_flows[(name, connet2name)]).sum(axis=-1,keepdims=False).reshape(H_in, W_in)
            activation_gradients[connet2name] = (gradient_flows[(name, connet2name)]).sum(axis=0,keepdims=False).reshape(H_out, W_out)
    return hook

model.conv1.register_backward_hook(get_activation_grad(None, "conv1"))
model.conv2.register_backward_hook(get_activation_grad("conv1", "conv2"))
model.conv3.register_backward_hook(get_activation_grad("conv2", "conv3"))

#########################################
# 3. Run forward/backward on a random input
#########################################
input_tensor = torch.randn(1, 1, 4, 4)
target = torch.randn(1, 1, 4, 4).long()
optimizer.zero_grad()
a3 = model(input_tensor)
loss = (a3.reshape(1,-1) - target[0,0,2,2].reshape(1,-1)).mean()  # use conv3 output for loss
loss.backward()

c:\Users\Admin\miniconda3\envs\data_science\lib\site-packages\torch\nn\modules\module.py:1344: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "


Save gradient flow information, includes: 
- activation_gradients: dict of aggregated weights of nodes at each layer. 
- gradient_flows: dict of values of gradient flows between adjacent layers.

In [2]:
import pickle
flow_info = {"input_representation": input_tensor[0,0,:,:].cpu().numpy(), 
             "activation_gradients": activation_gradients, 
             "gradient_flows": gradient_flows}
with open('flow_info.pkl', 'wb') as f:
    pickle.dump(flow_info, f)
np.save('target_representation.npy', target[0,0,:,:].cpu().numpy())

# Rendering the Visualization

Load gradient information

In [3]:
import pickle
import numpy as np

debug_folder = "./"
input_fpath = "input_representation.npy"
with open(debug_folder + 'flow_info.pkl', 'rb') as file:
    flow_data = pickle.load(file)
layer_activation_gradients = flow_data["activation_gradients"]
inter_layer_gradient_flows = flow_data["gradient_flows"]
input_representation = np.load(debug_folder+input_fpath)
target_representation = np.load(debug_folder + 'target_representation.npy')

Viz

In [4]:
import pickle
import numpy as np
from torchvision import transforms
import plotly.graph_objs as go
import ipywidgets as widgets
from ipyevents import Event
from IPython.display import display
from matplotlib import cm as cm
from matplotlib import colors as mcolors
import math

# Create a Progress with a description
def create_progress_html(value, max_value=100, width_px=300, height_px=30):
    """
    Tạo đoạn HTML mô phỏng một thanh progress với phần trăm ở giữa.
    
    value:      giá trị hiện tại (số phần trăm).
    max_value:  giá trị tối đa (thường 100).
    width_px:   chiều rộng thanh progress.
    height_px:  chiều cao thanh progress.
    """
    # Tính % để hiển thị trong thanh màu
    percent_fill = (value / max_value) * 100

    # HTML cho thanh progress
    progress_html = f"""
    <div style="position: relative; width: {width_px}px; height: {height_px}px; background-color: #ddd;">
      <!-- Phần màu hiển thị progress -->
      <div style="
          position: absolute; 
          width: {percent_fill}%; 
          height: 100%; 
          background-color: #00bcd4;">
      </div>
      <!-- Phần text nằm chính giữa -->
      <div style="
          position: absolute; 
          width: 100%; 
          text-align: center; 
          line-height: {height_px}px; 
          font-weight: bold;">
        {int(value)}%
      </div>
    </div>
    """
    return progress_html

# Tạo một widget HTML
progress_widget = widgets.HTML()
display(progress_widget)

def set_progress(value):
    # Cập nhật HTML bên trong widget
    progress_widget.value = create_progress_html(value)

# Update progress (0%)
set_progress(0)


# -------------------------
# 1. Load Data
# -------------------------
with open(debug_folder + 'flow_info.pkl', 'rb') as file:
    flow_data = pickle.load(file)
layer_activation_gradients = flow_data["activation_gradients"]
inter_layer_gradient_flows = flow_data["gradient_flows"]
input_representation = np.load(debug_folder+input_fpath)
target_representation = np.load(debug_folder + 'target_representation.npy')

# Create a copy of the input representation for editing purposes
editable_input_representation = input_representation.copy()

# Update progress (5%)
set_progress(5)

# -------------------------
# 2. Process Gradients: Flatten and Map by Layer
# -------------------------
flattened_gradients_by_layer = {}
all_layers = []
selectable_layers = []
for layer_key, gradient_matrix in layer_activation_gradients.items():
    all_layers.append(layer_key)
    flattened_gradients_by_layer[layer_key] = gradient_matrix.reshape(-1)
for flow_key, flow_value in inter_layer_gradient_flows.items():
    all_layers.append(flow_key[0])
    all_layers.append(flow_key[1])
    if not isinstance(flow_value, dict):
        if flow_value is not None:
            if flow_key[0] not in flattened_gradients_by_layer:
                flattened_gradients_by_layer[flow_key[0]] = flow_value.sum(axis=-1, keepdims=True)
            if flow_key[1] not in flattened_gradients_by_layer:
                flattened_gradients_by_layer[flow_key[1]] = flow_value.sum(axis=0, keepdims=True)
    else:
        H, W = (layer_activation_gradients[flow_key[0]]).shape
        indices = list(flow_value["indices_in"])
        if not isinstance(indices[0], list):
            if (indices[0]).shape[0] < (H*W):
                indices0, indices1 = [], []
                for idx in indices:
                    indices0.append(idx//W)
                    indices1.append(idx%W)
                flow_value["indices_in"] = (indices0, indices1)
        H, W = (layer_activation_gradients[flow_key[1]]).shape
        indices = list(flow_value["indices_out"])
        if not isinstance(indices[0], list):
            if (indices[0]).shape[0] < (H*W):
                indices0, indices1 = [], []
                for idx in indices:
                    indices0.append(idx//W)
                    indices1.append(idx%W)
                flow_value["indices_out"] = (indices0, indices1)
    selectable_layers.append(flow_key[0])
    
all_layers = sorted(set(all_layers))
selectable_layers = sorted(set(selectable_layers))

# Initialize selected layer variables
selected_layer = selectable_layers[0]
connected_layer = selected_layer
selected_node_index = 0
for flow_key in inter_layer_gradient_flows.keys():
    if flow_key[0] == selected_layer:
        connected_layer = flow_key[1]
        
# Update progress (7%)
set_progress(7)

# -------------------------
# 3. Compute Flow Matrices
# -------------------------
def compute_flow_matrices(gradient_flows, gradients_by_layer):
    flow_matrices = {}
    for source_layer, source_gradient in gradients_by_layer.items():
        for target_layer, target_gradient in gradients_by_layer.items():
            layer_pair = (source_layer, target_layer)
            if layer_pair in gradient_flows:
                flow_value = gradient_flows[layer_pair]
                if flow_value is None:
                    divisor = (0.*source_gradient.reshape(-1, 1)+1.) * target_gradient.reshape(1, -1)
                    flow_value = source_gradient.reshape(-1, 1) * target_gradient.reshape(1, -1)
                    flow_value[divisor!=0] = flow_value[divisor!=0] / divisor[divisor!=0] / divisor[divisor!=0]
                    divisor = (0.*flow_value+1.) * np.sum(np.abs(flow_value), axis=-1, keepdims=True)
                    flow_value = flow_value * np.sum(np.abs(flow_value), axis=-1, keepdims=True)
                    flow_value[divisor!=0] = flow_value[divisor!=0] / divisor[divisor!=0] / divisor[divisor!=0]
                flow_matrices[layer_pair] = flow_value
    return flow_matrices

flow_matrices = compute_flow_matrices(inter_layer_gradient_flows, flattened_gradients_by_layer)

# Determine dynamic threshold limits for the flow slider
flow_threshold_min, flow_threshold_max = None, None
for key, matrix in flow_matrices.items():
    if matrix is not None:
        if isinstance(matrix, dict):
            if flow_threshold_min is None or 0 < flow_threshold_min:
                flow_threshold_min = 0
            if flow_threshold_max is None or 1 > flow_threshold_max:
                flow_threshold_max = 1
        else:
            if flow_threshold_min is None or matrix.min() < flow_threshold_min:
                flow_threshold_min = matrix.min()
            if flow_threshold_max is None or matrix.max() > flow_threshold_max:
                flow_threshold_max = matrix.max()
if flow_threshold_min is None:
    flow_threshold_min = -1
if flow_threshold_max is None:
    flow_threshold_max = -1
    
# Update progress (10%)
set_progress(10)

# -------------------------
# 4. Build Sankey Diagram Data
# -------------------------
global_node_indices = {}  # Mapping: (layer, (row, col)) -> unique global node index
current_global_index = 0

layer_color_maps = {}
color_toggle_counter = 0
all_node_colors = []
for layer in all_layers:
    gradient_vector = flattened_gradients_by_layer[layer]
    if color_toggle_counter % 2:
        color_list = [mcolors.to_hex(cm.viridis(i / len(gradient_vector))) for i in range(len(gradient_vector))]
    else:
        color_list = [mcolors.to_hex(cm.plasma(i / len(gradient_vector))) for i in range(len(gradient_vector))]
    layer_color_maps[layer] = color_list
    color_toggle_counter += 1
    all_node_colors += color_list

layer_node_positions = {}  # Mapping for each layer: index -> (row, col)
for i, layer_label in enumerate(all_layers):
    H, W = layer_activation_gradients[layer_label].shape[-2:]
    layer_node_positions[layer_label] = [(r, c) for r in range(H) for c in range(W)]
    for j, (row, col) in enumerate(layer_node_positions[layer_label]):
        global_node_indices[(layer_label, (row, col))] = current_global_index
        current_global_index += 1
    
# Update progress (18%)
set_progress(18)

# Build lists for Sankey diagram: sources, targets, and flow values
sankey_sources, sankey_targets, sankey_values = [], [], []
for layer_pair, flow_matrix in flow_matrices.items():
    source_layer, target_layer = layer_pair
    source_positions = layer_node_positions[source_layer]
    target_positions = layer_node_positions[target_layer]
    flow_value = 0
    if flow_matrix is not None:
        if isinstance(flow_matrix, dict):
            indices_in = (flow_matrix["indices_in"])
            indices_out = (flow_matrix["indices_out"])
            id_in = 0
            id_out = 0
            for i, (r1, c1) in enumerate(source_positions):
                for j, (r2, c2) in enumerate(target_positions):
                    if flow_value > 0:
                        while id_in<len(indices_in) and ((indices_in[0])[id_in]) < r1 and ((indices_in[1])[id_in]) < c1:
                            id_in += 1
                        while id_out<len(indices_out) and ((indices_out[0])[id_out]) < r2 and ((indices_out[1])[id_out]) < c2:
                            id_out += 1
                        if (id_in<len(indices_in) and ((indices_in[0])[id_in]) == r1 and ((indices_in[1])[id_in]) == c1) or (id_out<len(indices_out) and ((indices_out[0])[id_out]) == r2 and ((indices_out[1])[id_out]) == c2):
                            sankey_sources.append(global_node_indices[(source_layer, (r1, c1))])
                            sankey_targets.append(global_node_indices[(target_layer, (r2, c2))])
                            sankey_values.append(1)
        else:
            for i, (r1, c1) in enumerate(source_positions):
                for j, (r2, c2) in enumerate(target_positions):
                    flow_value = flow_matrix[i, j]
                    if flow_value > 0:
                        sankey_sources.append(global_node_indices[(source_layer, (r1, c1))])
                        sankey_targets.append(global_node_indices[(target_layer, (r2, c2))])
                        sankey_values.append(flow_value)
                
region_settings = {}
region_settings[selected_layer] = [0, int(layer_activation_gradients[selected_layer].shape[-1] - 1), 0, int(layer_activation_gradients[selected_layer].shape[-2] - 1)]
region_settings[connected_layer] = [0, int(layer_activation_gradients[connected_layer].shape[-1] - 1), 0, int(layer_activation_gradients[connected_layer].shape[-2] - 1)]

# Update progress (25%)
set_progress(25)


# -------------------------
# 5. Widgets for Offset and Region Selection
# -------------------------
# Offsets for target and input representations
z_value_widget = widgets.IntText(value=1., description='Z:', layout=widgets.Layout(width='100px'), style={'description_width': '15px'})
target_row_offset_widget = widgets.IntText(value=0, description='Target Row Offset:', layout=widgets.Layout(width='200px'), style={'description_width': '120px'})
target_col_offset_widget = widgets.IntText(value=0, description='Target Col Offset:', layout=widgets.Layout(width='200px'), style={'description_width': '120px'})
input_row_offset_widget  = widgets.IntText(value=0, description='Input Row Offset:',  layout=widgets.Layout(width='200px'), style={'description_width': '120px'})
input_col_offset_widget  = widgets.IntText(value=0, description='Input Col Offset:',  layout=widgets.Layout(width='200px'), style={'description_width': '120px'})

# Region selection widgets for the Select (Conv1/Input) heatmap
select_xmin_widget = widgets.IntText(value=(region_settings[selected_layer])[0], description='Select Xmin:', layout=widgets.Layout(width='150px'))
select_xmax_widget = widgets.IntText(value=(region_settings[selected_layer])[1], description='Select Xmax:', layout=widgets.Layout(width='150px'))
select_ymin_widget = widgets.IntText(value=(region_settings[selected_layer])[2], description='Select Ymin:', layout=widgets.Layout(width='150px'))
select_ymax_widget = widgets.IntText(value=(region_settings[selected_layer])[3], description='Select Ymax:', layout=widgets.Layout(width='150px'))
select_region_box = widgets.HBox([select_xmin_widget, select_xmax_widget, select_ymin_widget, select_ymax_widget])

# Region selection widgets for the Output (Target) heatmap
output_xmin_widget = widgets.IntText(value=(region_settings[connected_layer])[0], description='Output Xmin:', layout=widgets.Layout(width='150px'))
output_xmax_widget = widgets.IntText(value=(region_settings[connected_layer])[1], description='Output Xmax:', layout=widgets.Layout(width='150px'))
output_ymin_widget = widgets.IntText(value=(region_settings[connected_layer])[2], description='Output Ymin:', layout=widgets.Layout(width='150px'))
output_ymax_widget = widgets.IntText(value=(region_settings[connected_layer])[3], description='Output Ymax:', layout=widgets.Layout(width='150px'))
output_region_box = widgets.HBox([output_xmin_widget, output_xmax_widget, output_ymin_widget, output_ymax_widget])

# Apply Regions button for both region groups
apply_regions_button = widgets.Button(description="Apply Regions", button_style='warning')

# Update progress (35%)
set_progress(35)

# -------------------------
# 6. Functions to Generate Plotly Figures
# -------------------------
def create_sankey(selected_layer, selected_node_index, threshold_value):
    new_node_labels, new_node_x, new_node_y, new_node_colors, new_link_colors = [], [], [], [], []
    new_sources, new_targets, new_values = [], [], []
    sankey_figure = None
    if sankey_container is None or sankey_container.layout.display == 'none':
        sankey_figure = go.FigureWidget(data=[go.Sankey(
        arrangement='fixed',
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color='black', width=0.5),
            label=new_node_labels,
            color=new_node_colors,
            x=new_node_x,
            y=new_node_y
        ),
        link=dict(
            source=new_sources,
            target=new_targets,
            value=new_values,
            color=new_link_colors
        )
    )])
    else:
        new_index_mapping = {}
        label_counter = 0
        for i, layer in enumerate(all_layers):
            H, W = layer_activation_gradients[layer].shape[-2:]
            layer_node_positions[layer] = [(r, c) for r in range(H) for c in range(W)]
            node_color_list = layer_color_maps[layer]
            updated_node_colors = []
            inner_counter = 0
            for j, (row, col) in enumerate(layer_node_positions[layer]):
                global_index = global_node_indices[(layer, (row, col))]
                if (layer not in region_settings) or ((region_settings[layer])[0] <= col <= (region_settings[layer])[1] and (region_settings[layer])[2] <= (H - 1 - row) <= (region_settings[layer])[3]):
                    new_node_labels.append(f"{layer}_{row},{col}")
                    new_node_x.append((float(i) + 0.01) / (len(all_layers) - 0.55))
                    if layer in region_settings:
                        denom = ((region_settings[layer])[1] - (region_settings[layer])[0] + 1) * ((region_settings[layer])[3] - (region_settings[layer])[2] + 1)
                        new_node_y.append((float(inner_counter) + 0.5) / denom)
                    else:
                        new_node_y.append((float(inner_counter) + 0.5) / (len(layer_node_positions[layer])))
                    updated_node_colors.append("red" if j == selected_node_index else node_color_list[j])
                    new_index_mapping[global_index] = label_counter
                    inner_counter += 1
                    label_counter += 1
            new_node_colors += updated_node_colors

        for idx, src in enumerate(sankey_sources):
            if src in new_index_mapping and sankey_targets[idx] in new_index_mapping:
                new_sources.append(new_index_mapping[src])
                new_targets.append(new_index_mapping[sankey_targets[idx]])
                new_values.append(sankey_values[idx])
                if src == global_node_indices[(selected_layer, layer_node_positions[selected_layer][selected_node_index])] and float(sankey_values[idx]) > threshold_value:
                    new_link_colors.append("red")
                else:
                    new_link_colors.append(f"rgba({128*int(float(sankey_values[idx])>threshold_value)},"
                                             f"{128*int(float(sankey_values[idx])>threshold_value)},"
                                             f"{128*int(float(sankey_values[idx])>threshold_value)},"
                                             f"{1.*int(float(sankey_values[idx])>threshold_value)+.1})")
        sankey_figure = go.FigureWidget(data=[go.Sankey(
            arrangement='fixed',
            node=dict(
                pad=15,
                thickness=20,
                line=dict(color='black', width=0.5),
                label=new_node_labels,
                color=new_node_colors,
                x=new_node_x,
                y=new_node_y
            ),
            link=dict(
                source=new_sources,
                target=new_targets,
                value=new_values,
                color=new_link_colors
            )
        )])
        sankey_figure.update_layout(title_text='Gradient Flow', font_size=10)
    return sankey_figure

def create_heatmap_conv1(selected_node_index, selected_layer):
    highlight_shapes = []
    grid_shape = layer_activation_gradients[selected_layer].shape
    if selected_node_index is not None:
        row, col = divmod(selected_node_index, grid_shape[-1])
        highlight_shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - 0.5, y0=grid_shape[0] - 1 - row - 0.5,
            x1=col + 0.5, y1=grid_shape[0] - 1 - row + 0.5,
            line=dict(color='red', width=3)
        ))
    heatmap_conv1 = go.FigureWidget(data=go.Heatmap(
        z=(layer_activation_gradients[selected_layer])[::-1, :],
        colorscale='Viridis',
        zmin=layer_activation_gradients[selected_layer].min(),
        zmax=layer_activation_gradients[selected_layer].max(),
        colorbar=dict(title='', len=0.5, x=1.02, xanchor='left', thickness=10)
    ))
    try:
        xmin, xmax = ((region_settings[selected_layer])[0]) - 0.5, ((region_settings[selected_layer])[1]) + 0.5
        ymin, ymax = ((region_settings[selected_layer])[2]) - 0.5, ((region_settings[selected_layer])[3]) + 0.5
        heatmap_conv1.update_layout(
            xaxis=dict(range=[xmin, xmax]),
            yaxis=dict(range=[ymin, ymax])
        )
    except Exception as e:
        pass

    def on_x_range_change(layout, x_range):
        select_xmin_widget.value = int(x_range[0])
        select_xmax_widget.value = int(x_range[1])
    def on_y_range_change(change, y_range):
        select_ymin_widget.value = int(y_range[0])
        select_ymax_widget.value = int(y_range[1])
    heatmap_conv1.layout.on_change(on_x_range_change, 'xaxis.range')
    heatmap_conv1.layout.on_change(on_y_range_change, 'yaxis.range')

    heatmap_conv1.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=highlight_shapes,
        annotations=[dict(
            text="Selected-Layer Gradient",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return heatmap_conv1

def create_heatmap_conv2(selected_node_index, selected_layer, threshold_value):
    global connected_layer
    highlight_shapes = []

    grid_shape = layer_activation_gradients[connected_layer].shape
    if not isinstance(flow_matrices[(selected_layer, connected_layer)], dict):
        flow_matrix_thresholded = None
        if flow_matrices[(selected_layer, connected_layer)] is not None:
            flow_matrix_thresholded = ((flow_matrices[(selected_layer, connected_layer)] > threshold_value)[selected_node_index, :]).reshape(*grid_shape)
        for r in range(grid_shape[0]):
            for c in range(grid_shape[1]):
                if (flow_matrix_thresholded is not None) and flow_matrix_thresholded[r, c] > 0:
                    highlight_shapes.append(dict(
                        type='circle', xref='x', yref='y',
                        x0=c - 0.5, y0=grid_shape[0] - 1 - r - 0.5,
                        x1=c + 0.5, y1=grid_shape[0] - 1 - r + 0.5,
                        line=dict(color='red', width=3)
                    ))
    heatmap_conv2 = go.FigureWidget(data=go.Heatmap(
        z=(layer_activation_gradients[connected_layer])[::-1, :],
        colorscale='Viridis',
        zmin=layer_activation_gradients[connected_layer].min(),
        zmax=layer_activation_gradients[connected_layer].max(),
        colorbar=dict(title='', len=0.5, x=1.02, xanchor='left', thickness=10)
    ))
    try:
        xmin, xmax = ((region_settings[connected_layer])[0]) - 0.5, ((region_settings[connected_layer])[1]) + 0.5
        ymin, ymax = ((region_settings[connected_layer])[2]) - 0.5, ((region_settings[connected_layer])[3]) + 0.5
        heatmap_conv2.update_layout(
            xaxis=dict(range=[xmin, xmax]),
            yaxis=dict(range=[ymin, ymax])
        )
    except Exception as e:
        pass

    def on_x_range_change(layout, x_range):
        output_xmin_widget.value = int(x_range[0])
        output_xmax_widget.value = int(x_range[1])
    def on_y_range_change(change, y_range):
        output_ymin_widget.value = int(y_range[0])
        output_ymax_widget.value = int(y_range[1])
    heatmap_conv2.layout.on_change(on_x_range_change, 'xaxis.range')
    heatmap_conv2.layout.on_change(on_y_range_change, 'yaxis.range')

    heatmap_conv2.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=highlight_shapes,
        annotations=[dict(
            text="Output-Layer Gradient",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return heatmap_conv2

def create_heatmap_target(selected_node_index):
    row_offset = -target_row_offset_widget.value
    col_offset = target_col_offset_widget.value
    highlight_shapes = []
    grid_shape = target_representation.shape
    if selected_node_index is not None:
        row, col = divmod(selected_node_index, layer_activation_gradients[selected_layer].shape[-1])
        highlight_shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - col_offset - 0.5, y0=grid_shape[0] - 1 - row - row_offset - 0.5,
            x1=col - col_offset + 0.5, y1=grid_shape[0] - 1 - row - row_offset + 0.5,
            line=dict(color='red', width=3)
        ))
    target_heatmap = go.FigureWidget(data=go.Heatmap(
        z=target_representation[::-1, :],
        colorscale='Viridis',
        zmin=target_representation.min(),
        zmax=target_representation.max(),
        colorbar=dict(title='', len=0.5, x=1.02, xanchor='left', thickness=10)
    ))
    try:
        xmin, xmax = ((region_settings[selected_layer])[0]) - col_offset - 0.5, ((region_settings[selected_layer])[1]) - col_offset + 0.5
        ymin, ymax = ((grid_shape[0] - layer_activation_gradients[selected_layer].shape[0]) + ((region_settings[selected_layer])[2]) - row_offset - 0.5), ((grid_shape[0] - layer_activation_gradients[selected_layer].shape[0]) + ((region_settings[selected_layer])[3]) - row_offset + 0.5)
        target_heatmap.update_layout(
            xaxis=dict(range=[xmin, xmax]),
            yaxis=dict(range=[ymin, ymax])
        )
    except Exception as e:
        pass
    target_heatmap.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=highlight_shapes,
        annotations=[dict(
            text="Target Representation",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return target_heatmap

def create_heatmap_input(selected_node_index):
    input_row_offset_value = -input_row_offset_widget.value
    input_col_offset_value = input_col_offset_widget.value
    highlight_shapes = []
    grid_shape = editable_input_representation.shape
    if selected_node_index is not None:
        row, col = divmod(selected_node_index, layer_activation_gradients[selected_layer].shape[-1])
        highlight_shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - input_col_offset_value - 0.5, y0=grid_shape[0] - 1 - row - input_row_offset_value - 0.5,
            x1=col - input_col_offset_value + 0.5, y1=grid_shape[0] - 1 - row - input_row_offset_value + 0.5,
            line=dict(color='red', width=3)
        ))
    input_heatmap = go.FigureWidget(data=go.Heatmap(
        z=editable_input_representation[::-1, :],
        colorscale=[[0, 'white'], [1, 'black']],
        zmin=0, zmax=1,
        colorbar=dict(title='', len=0.5, x=1.02, xanchor='left', thickness=10)
    ))
    try:
        xmin, xmax = ((region_settings[selected_layer])[0]) - input_col_offset_value - 0.5, ((region_settings[selected_layer])[1]) - input_col_offset_value + 0.5
        ymin, ymax = ((grid_shape[0] - layer_activation_gradients[selected_layer].shape[0]) + ((region_settings[selected_layer])[2]) - input_row_offset_value - 0.5), ((grid_shape[0] - layer_activation_gradients[selected_layer].shape[0]) + ((region_settings[selected_layer])[3]) - input_row_offset_value + 0.5)
        input_heatmap.update_layout(
            xaxis=dict(range=[xmin, xmax]),
            yaxis=dict(range=[ymin, ymax])
        )
    except Exception as e:
        pass
    input_heatmap.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=highlight_shapes,
        annotations=[dict(
            text="Input Representation",
            x=0.5 + input_col_offset_value / 100.0, y=-0.15 + input_row_offset_value / 100.0,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return input_heatmap

def create_all_heatmaps(selected_node_index, selected_layer, threshold_value):
    heatmap_conv1 = create_heatmap_conv1(selected_node_index, selected_layer)
    heatmap_conv2 = create_heatmap_conv2(selected_node_index, selected_layer, threshold_value)
    target_heatmap = create_heatmap_target(selected_node_index)
    input_heatmap = create_heatmap_input(selected_node_index)
    return heatmap_conv1, heatmap_conv2, target_heatmap, input_heatmap

# Update progress (37%)
set_progress(37)


# -------------------------
# 7. Callbacks for Conv1 Heatmap, Input Heatmap and Region Reset
# -------------------------
def conv1_on_click(trace, points, state):
    if points.point_inds:
        total_rows, total_cols = editable_input_representation.shape
        # points.point_inds returns a list of indices; extract first index (as a tuple)
        clicked_row_display = (points.point_inds[0])[0]
        clicked_col = (points.point_inds[0])[1]
        actual_row = total_rows - 1 - clicked_row_display
        selected_node_index = global_node_indices[(selected_layer, (actual_row, clicked_col))]
        node_slider.value = selected_node_index

def input_on_click(trace, points, state):
    if points.point_inds:
        total_rows, total_cols = editable_input_representation.shape
        # points.point_inds returns a list of indices; extract first index (as a tuple)
        clicked_row_display = (points.point_inds[0])[0]
        clicked_col = (points.point_inds[0])[1]
        actual_row = total_rows - 1 - clicked_row_display
        editable_input_representation[actual_row, clicked_col] = 0.0 if editable_input_representation[actual_row, clicked_col] > 0 else z_value_widget.value
        trace.z = editable_input_representation[::-1, :]
        save_button.disabled = False
        
def on_apply_regions_clicked(button):
    global selected_layer, connected_layer, selected_node_index, current_threshold
    selected_layer = layer_dropdown.value
    connected_layer = selected_layer
    for flow_key in inter_layer_gradient_flows.keys():
        if flow_key[0] == selected_layer:
            connected_layer = flow_key[1]
    update_slider_range(selected_layer)
    current_threshold = threshold_slider.value
    selected_node_index = node_slider.value
    
    if not (selected_layer in region_settings):
        region_settings[selected_layer] = [0, int(layer_activation_gradients[selected_layer].shape[-1] - 1), 0, int(layer_activation_gradients[selected_layer].shape[-2] - 1)]
    if button is not None:
        region_settings[selected_layer] = (select_xmin_widget.value, select_xmax_widget.value, select_ymin_widget.value, select_ymax_widget.value)
    
    if not (connected_layer in region_settings):
        region_settings[connected_layer] = [0, int(layer_activation_gradients[connected_layer].shape[-1] - 1), 0, int(layer_activation_gradients[connected_layer].shape[-2] - 1)]
    if button is not None:
        region_settings[connected_layer] = (output_xmin_widget.value, output_xmax_widget.value, output_ymin_widget.value, output_ymax_widget.value)


    if sankey_container.layout.display != 'none':
        sankey_toggle.description = 'Expand Sankey Diagram'
        sankey_container.layout.display = 'none'
    new_sankey_fig = create_sankey(selected_layer, selected_node_index, current_threshold)
    sankey_widget.data[0].node.label = new_sankey_fig.data[0].node.label
    sankey_widget.data[0].node.color = new_sankey_fig.data[0].node.color
    sankey_widget.data[0].node.x = new_sankey_fig.data[0].node.x
    sankey_widget.data[0].node.y = new_sankey_fig.data[0].node.y
    sankey_widget.data[0].link.source = new_sankey_fig.data[0].link.source
    sankey_widget.data[0].link.target = new_sankey_fig.data[0].link.target
    sankey_widget.data[0].link.value = new_sankey_fig.data[0].link.value
    sankey_widget.data[0].link.color = new_sankey_fig.data[0].link.color

    global heatmap_conv1_fig, heatmap_conv2_fig, target_heatmap_fig, input_heatmap_fig
    heatmap_conv1_fig, heatmap_conv2_fig, target_heatmap_fig, input_heatmap_fig = create_all_heatmaps(node_slider.value, layer_dropdown.value, threshold_slider.value)
    heatmap_conv1_fig.data[0].on_click(conv1_on_click)
    input_heatmap_fig.data[0].on_click(input_on_click)

    with heatmap_conv1_widget:
        heatmap_conv1_widget.clear_output(wait=True)
        display(heatmap_conv1_fig)
    with heatmap_conv2_widget:
        heatmap_conv2_widget.clear_output(wait=True)
        display(heatmap_conv2_fig)
    with target_widget:
        target_widget.clear_output(wait=True)
        display(target_heatmap_fig)
    with input_widget:
        input_widget.clear_output(wait=True)
        display(input_heatmap_fig)
        
# Update progress (40%)
set_progress(40)
        
# -------------------------
# 8. IPyWidgets for Interaction and Buttons
# -------------------------
layer_dropdown = widgets.Dropdown(
    options=selectable_layers,
    value=selected_layer,
    description='Select Layer:'
)

node_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=int(np.prod(layer_activation_gradients[layer_dropdown.value].shape[-2:])) - 1,
    step=1,
    description='Node Index:'
)

current_threshold = (flow_threshold_max - flow_threshold_min) / 2
threshold_slider = widgets.FloatSlider(
    value=current_threshold,
    min=flow_threshold_min,
    max=flow_threshold_max,
    step=(flow_threshold_max - flow_threshold_min) / 100,
    description='Threshold:'
)

def update_slider_range(selected_layer):
    node_slider.min = 0
    node_slider.max = int(np.prod(layer_activation_gradients[selected_layer].shape[-2:])) - 1
    if node_slider.value > node_slider.max:
        node_slider.value = 0
    threshold_slider.min = flow_threshold_min
    threshold_slider.max = flow_threshold_max
    if threshold_slider.value > threshold_slider.max:
        threshold_slider.value = (flow_threshold_max - flow_threshold_min) / 2

# Render and Save buttons
render_button = widgets.Button(description="Render", button_style='primary')
save_button = widgets.Button(description="Save", button_style='primary', disabled=True)

def on_render_button_clicked(button):
    save_button.disabled = True
    with open(debug_folder + 'flow_info.pkl', 'rb') as file:
        flow_data = pickle.load(file)
    global input_representation, editable_input_representation, layer_activation_gradients, inter_layer_gradient_flows, target_representation
    layer_activation_gradients = flow_data["activation_gradients"]
    inter_layer_gradient_flows = flow_data["gradient_flows"]
    input_representation = np.load(debug_folder + input_fpath)
    target_representation = np.load(debug_folder + 'target_representation.npy')
    
    editable_input_representation = input_representation.copy()
    on_interaction_change(None)

def on_save_button_clicked(button):
    global input_fpath
    if input_fpath_text.value != "":
        input_fpath = input_fpath_text.value
    save_button.disabled = True
    flow_data = {
        "activation_gradients": layer_activation_gradients,
        "gradient_flows": inter_layer_gradient_flows
    }
    np.save(debug_folder+input_fpath, editable_input_representation)
    with open(debug_folder + 'flow_info.pkl', 'wb') as file:
        pickle.dump(flow_data, file)

apply_regions_button.on_click(on_apply_regions_clicked)
render_button.on_click(on_render_button_clicked)
save_button.on_click(on_save_button_clicked)

# Update progress (55%)
set_progress(55)


# -------------------------
# 9. Expand/Collapse for Sankey Diagram
# -------------------------
sankey_toggle = widgets.ToggleButton(
    value=False,
    description='Expand Sankey Diagram',
    button_style='info',
    tooltip='Click to expand/collapse the Sankey Diagram'
)

sankey_container = None
sankey_widget = create_sankey(layer_dropdown.value, node_slider.value, threshold_slider.value)
sankey_container = widgets.VBox([sankey_widget])
sankey_container.layout.display = 'none'

def on_sankey_toggle_change(change):
    if change['new']:
        sankey_toggle.description = 'Collapse Sankey Diagram'
        sankey_container.layout.display = 'flex'
    else:
        sankey_toggle.description = 'Expand Sankey Diagram'
        sankey_container.layout.display = 'none'
    new_sankey_fig = create_sankey(selected_layer, selected_node_index, current_threshold)
    sankey_widget.data[0].node.label = new_sankey_fig.data[0].node.label
    sankey_widget.data[0].node.color = new_sankey_fig.data[0].node.color
    sankey_widget.data[0].node.x = new_sankey_fig.data[0].node.x
    sankey_widget.data[0].node.y = new_sankey_fig.data[0].node.y
    sankey_widget.data[0].link.source = new_sankey_fig.data[0].link.source
    sankey_widget.data[0].link.target = new_sankey_fig.data[0].link.target
    sankey_widget.data[0].link.value = new_sankey_fig.data[0].link.value
    sankey_widget.data[0].link.color = new_sankey_fig.data[0].link.color
        

sankey_toggle.observe(on_sankey_toggle_change, names='value')

# Update progress (75%)
set_progress(75)

# -------------------------
# 10. Layout and Update Functions
# -------------------------
# Initialize global heatmap widgets
heatmap_conv1_widget = widgets.Output()
heatmap_conv2_widget = widgets.Output()
target_widget = widgets.Output()
input_widget = widgets.Output()

heatmap_conv1_fig, heatmap_conv2_fig, target_heatmap_fig, input_heatmap_fig = create_all_heatmaps(node_slider.value, layer_dropdown.value, threshold_slider.value)
heatmap_conv1_fig.data[0].on_click(conv1_on_click)
input_heatmap_fig.data[0].on_click(input_on_click)

with heatmap_conv1_widget:
    heatmap_conv1_widget.clear_output(wait=True)
    display(heatmap_conv1_fig)
with heatmap_conv2_widget:
    heatmap_conv2_widget.clear_output(wait=True)
    display(heatmap_conv2_fig)
with target_widget:
    target_widget.clear_output(wait=True)
    display(target_heatmap_fig)
with input_widget:
    input_widget.clear_output(wait=True)
    display(input_heatmap_fig)

def update_figures(selected_layer, selected_node_index, threshold_value):
    new_sankey_fig = create_sankey(selected_layer, selected_node_index, threshold_value)
    sankey_widget.data[0].node.label = new_sankey_fig.data[0].node.label
    sankey_widget.data[0].node.color = new_sankey_fig.data[0].node.color
    sankey_widget.data[0].node.x = new_sankey_fig.data[0].node.x
    sankey_widget.data[0].node.y = new_sankey_fig.data[0].node.y
    sankey_widget.data[0].link.source = new_sankey_fig.data[0].link.source
    sankey_widget.data[0].link.target = new_sankey_fig.data[0].link.target
    sankey_widget.data[0].link.value = new_sankey_fig.data[0].link.value
    sankey_widget.data[0].link.color = new_sankey_fig.data[0].link.color
    
    new_heatmap_conv1, new_heatmap_conv2, new_target_heatmap, new_input_heatmap = create_all_heatmaps(selected_node_index, selected_layer, threshold_value)
    
    heatmap_conv1_fig.data[0].z = new_heatmap_conv1.data[0].z
    heatmap_conv1_fig.layout.title.text = new_heatmap_conv1.layout.title.text
    heatmap_conv1_fig.layout.shapes = new_heatmap_conv1.layout.shapes
    heatmap_conv1_fig.layout.annotations = new_heatmap_conv1.layout.annotations

    heatmap_conv2_fig.data[0].z = new_heatmap_conv2.data[0].z
    heatmap_conv2_fig.layout.title.text = new_heatmap_conv2.layout.title.text
    heatmap_conv2_fig.layout.shapes = new_heatmap_conv2.layout.shapes
    heatmap_conv2_fig.layout.annotations = new_heatmap_conv2.layout.annotations

    target_heatmap_fig.data[0].z = new_target_heatmap.data[0].z
    target_heatmap_fig.layout.title.text = new_target_heatmap.layout.title.text
    target_heatmap_fig.layout.shapes = new_target_heatmap.layout.shapes
    target_heatmap_fig.layout.annotations = new_target_heatmap.layout.annotations
    
    input_heatmap_fig.data[0].z = new_input_heatmap.data[0].z
    input_heatmap_fig.layout.title.text = new_input_heatmap.layout.title.text
    input_heatmap_fig.layout.shapes = new_input_heatmap.layout.shapes
    input_heatmap_fig.layout.annotations = new_input_heatmap.layout.annotations

def on_interaction_change(change):
    global selected_layer, connected_layer, selected_node_index, current_threshold
    
    selected_layer = layer_dropdown.value
    if change is not None and 'new' in change and change['new'] in selectable_layers:
        connected_layer = selected_layer
        for flow_key in inter_layer_gradient_flows.keys():
            if flow_key[0] == selected_layer:
                connected_layer = flow_key[1]
        on_apply_regions_clicked(None)
        
        select_xmin_widget.value = int((region_settings[selected_layer])[0])
        select_xmax_widget.value = int((region_settings[selected_layer])[1])
        select_ymin_widget.value = int((region_settings[selected_layer])[2])
        select_ymax_widget.value = int((region_settings[selected_layer])[3])

        output_xmin_widget.value = int((region_settings[connected_layer])[0])
        output_xmax_widget.value = int((region_settings[connected_layer])[1])
        output_ymin_widget.value = int((region_settings[connected_layer])[2])
        output_ymax_widget.value = int((region_settings[connected_layer])[3])
    else:
        update_slider_range(selected_layer)
        current_threshold = threshold_slider.value
        selected_node_index = node_slider.value
        update_figures(selected_layer, selected_node_index, current_threshold)

apply_regions_button.on_click(on_apply_regions_clicked)
layer_dropdown.observe(on_interaction_change, names='value')
node_slider.observe(on_interaction_change, names='value')
threshold_slider.observe(on_interaction_change, names='value')

def on_select_region_change(change):
    selected_layer = layer_dropdown.value
    current_node_index = node_slider.value
    current_threshold = threshold_slider.value
    update_figures(selected_layer, current_node_index, current_threshold)

select_xmin_widget.observe(on_select_region_change, names='value')
select_xmax_widget.observe(on_select_region_change, names='value')
select_ymin_widget.observe(on_select_region_change, names='value')
select_ymax_widget.observe(on_select_region_change, names='value')

def on_output_region_change(change):
    selected_layer = layer_dropdown.value
    current_node_index = node_slider.value
    current_threshold = threshold_slider.value
    update_figures(selected_layer, current_node_index, current_threshold)

output_xmin_widget.observe(on_output_region_change, names='value')
output_xmax_widget.observe(on_output_region_change, names='value')
output_ymin_widget.observe(on_output_region_change, names='value')
output_ymax_widget.observe(on_output_region_change, names='value')

# Update progress (98%)
set_progress(98)

# -------------------------
# 11. Layout the UI
# -------------------------
output_heatmaps_box = widgets.HBox([heatmap_conv1_widget, heatmap_conv2_widget, target_widget])
input_fpath_text = widgets.Text(value="", description='Save to:', placeholder='input_representation.npy', layout=widgets.Layout(width='300px'), style={'description_width': '50px'})
def select_hidden_function(*args):
    select_xmin_widget.value = output_xmin_widget.value
    select_xmax_widget.value = output_xmax_widget.value
    select_ymin_widget.value = output_ymin_widget.value
    select_ymax_widget.value = output_ymax_widget.value
def output_hidden_function(*args):
    output_xmin_widget.value = select_xmin_widget.value
    output_xmax_widget.value = select_xmax_widget.value
    output_ymin_widget.value = select_ymin_widget.value
    output_ymax_widget.value = select_ymax_widget.value
select_hidden_html = widgets.HTML(value="""
    <div style="width:300px;height:30px;background-color:white;" tabindex="0">
      <b>Selected Layer Region:</b>
    </div>
    """)
output_hidden_html = widgets.HTML(value="""
    <div style="width:300px;height:30px;background-color:white;" tabindex="0">
      <b>Output Layer Region:</b>
    </div>
    """)
select_hidden_click_event = Event(source=select_hidden_html, watched_events=["click", "mousedown", "mouseup"])
select_hidden_click_event.on_dom_event(select_hidden_function)
output_hidden_click_event = Event(source=output_hidden_html, watched_events=["click", "mousedown", "mouseup"])
output_hidden_click_event.on_dom_event(output_hidden_function)
output_input_box = widgets.VBox([
    input_widget,
    input_fpath_text,
    widgets.HBox([render_button, save_button]),
    select_hidden_html,
    select_region_box,
    output_hidden_html,
    output_region_box,
    widgets.HTML(value="<b>Padding:</b>"),
    widgets.HBox([target_row_offset_widget, target_col_offset_widget]),
    widgets.HBox([input_row_offset_widget, input_col_offset_widget]),
    apply_regions_button
])
ui = widgets.VBox([
    sankey_toggle,
    sankey_container,
    widgets.HBox([layer_dropdown, node_slider, threshold_slider]),
    output_heatmaps_box,
    z_value_widget,
    output_input_box
])

# Update progress (100%)
set_progress(100)
progress_widget.close()  # This will remove the progress bar from display

display(ui)

HTML(value='')